# 10 Minutes to cuDF

Modelled after 10 Minutes to Pandas, this is a short introduction to cuDF, geared mainly towards new users.

## What are these Libraries?

[cuDF](https://github.com/NVIDIA/cudf) is a Python GPU DataFrame library (built on the Apache Arrow columnar memory format) for loading, joining, aggregating, filtering, and otherwise manipulating tabular data using a DataFrame style API in the style of [pandas](https://pandas.pydata.org).


## When to use cuDF

If your workflow is fast enough on a single GPU or your data comfortably fits in memory on a single GPU, cuDF is the right choice for GPU-accelerated DataFrame operations.

In [1]:
import os

import cudf
import cupy as cp
import pandas as pd

cp.random.seed(12)

#### Portions of this were borrowed and adapted from the
#### cuDF cheatsheet, existing cuDF documentation,
#### and 10 Minutes to Pandas.

## Object Creation

In [2]:
s = cudf.Series([1, 2, 3, None, 4])
s

0       1
1       2
2       3
3    <NA>
4       4
dtype: int64

In [3]:
df = cudf.DataFrame(
    {
        "a": list(range(20)),
        "b": list(reversed(range(20))),
        "c": list(range(20)),
    }
)
df

,a,b,c
0,0,19,0
1,1,18,1
2,2,17,2
3,3,16,3
4,4,15,4
5,5,14,5
6,6,13,6
7,7,12,7
8,8,11,8
9,9,10,9


In [4]:
pdf = pd.DataFrame({"a": [0, 1, 2, 3], "b": [0.1, 0.2, None, 0.3]})
gdf = cudf.DataFrame(pdf)
gdf

,a,b
0,0,0.1
1,1,0.2
2,2,NaN
3,3,0.3


## Viewing Data

Viewing the top rows of a GPU dataframe.

In [5]:
df.head(2)

,a,b,c
0,0,19,0
1,1,18,1


Sorting by values.

In [6]:
df.sort_values(by="b")

,a,b,c
19,19,0,19
18,18,1,18
17,17,2,17
16,16,3,16
15,15,4,15
14,14,5,14
13,13,6,13
12,12,7,12
11,11,8,11
10,10,9,10


## Selecting a Column

In [7]:
df["a"]

0      0
1      1
2      2
3      3
4      4
5      5
6      6
7      7
8      8
9      9
10    10
11    11
12    12
13    13
14    14
15    15
16    16
17    17
18    18
19    19
Name: a, dtype: int64

## Selecting Rows by Label

Selecting rows from index 2 to index 5 from columns 'a' and 'b'.

In [8]:
df.loc[2:5, ["a", "b"]]

,a,b
2,2,17
3,3,16
4,4,15
5,5,14


## Selecting Rows by Position

In [9]:
df.iloc[0]

a     0
b    19
c     0
Name: 0, dtype: int64

In [10]:
df.iloc[0:3, 0:2]

,a,b
0,0,19
1,1,18
2,2,17


You can also select elements of a `DataFrame` or `Series` with direct index access.

In [11]:
df[3:5]

,a,b,c
3,3,16,3
4,4,15,4


In [12]:
s[3:5]

3    <NA>
4       4
dtype: int64

## Boolean Indexing

Selecting rows in a `DataFrame` or `Series` by direct Boolean indexing.

In [13]:
df[df.b > 15]

,a,b,c
0,0,19,0
1,1,18,1
2,2,17,2
3,3,16,3


Selecting values from a `DataFrame` where a Boolean condition is met, via the `query` API.

In [14]:
df.query("b == 3")

,a,b,c
16,16,3,16


In [15]:
cudf_comparator = 3
df.query("b == @cudf_comparator")

,a,b,c
16,16,3,16


Using the `isin` method for filtering.

In [16]:
df[df.a.isin([0, 5])]

,a,b,c
0,0,19,0
5,5,14,5


## MultiIndex

cuDF supports hierarchical indexing of DataFrames using MultiIndex. Grouping hierarchically (see `Grouping` below) automatically produces a DataFrame with a MultiIndex.

In [17]:
arrays = [["a", "a", "b", "b"], [1, 2, 3, 4]]
tuples = list(zip(*arrays, strict=True))
idx = cudf.MultiIndex.from_tuples(tuples)
idx

MultiIndex([('a', 1),
            ('a', 2),
            ('b', 3),
            ('b', 4)],
           )

This index can back either axis of a DataFrame.

In [18]:
gdf1 = cudf.DataFrame(
    {"first": cp.random.rand(4), "second": cp.random.rand(4)}
)
gdf1.index = idx
gdf1

first    second
a 1  0.082654  0.967955
  2  0.399417  0.441425
b 3  0.784297  0.793582
  4  0.070303  0.271711

In [19]:
gdf2 = cudf.DataFrame(
    {"first": cp.random.rand(4), "second": cp.random.rand(4)}
).T
gdf2.columns = idx
gdf2

a                  b          
               1         2        3         4
first   0.343382  0.003700  0.20043  0.581614
second  0.907812  0.101512  0.24179  0.224180

Accessing values of a DataFrame with a MultiIndex, both with `.loc`

In [20]:
gdf1.loc[("b", 3)]

first     0.784297
second    0.793582
Name: ('b', 3), dtype: float64

And `.iloc`

In [21]:
gdf1.iloc[0:2]

first    second
a 1  0.082654  0.967955
  2  0.399417  0.441425

Missing Data
------------

Missing data can be replaced by using the `fillna` method.

In [22]:
s.fillna(999)

0      1
1      2
2      3
3    999
4      4
dtype: int64

## Stats

Calculating descriptive statistics for a `Series`.

In [23]:
s.mean(), s.var()

(np.float64(2.5), np.float64(1.6666666666666667))

## Applying Functions

In [24]:
def add_ten(num):
    return num + 10


df["a"].apply(add_ten)

0     10
1     11
2     12
3     13
4     14
5     15
6     16
7     17
8     18
9     19
10    20
11    21
12    22
13    23
14    24
15    25
16    26
17    27
18    28
19    29
Name: a, dtype: int64

## Histogramming

Counting the number of occurrences of each unique value of variable.

In [25]:
df.a.value_counts()

a
0     1
1     1
2     1
3     1
4     1
5     1
6     1
7     1
8     1
9     1
10    1
11    1
12    1
13    1
14    1
15    1
16    1
17    1
18    1
19    1
Name: count, dtype: int64

## String Methods

Like pandas, cuDF provides string processing methods in the `str` attribute of `Series`. Full documentation of string methods is a work in progress. Please see the [cuDF API documentation](https://docs.nvidia.com/cudf/latest/cudf/api_docs/series/#string-handling) for more information.

In [26]:
s = cudf.Series(["A", "B", "C", "Aaba", "Baca", None, "CABA", "dog", "cat"])
s.str.lower()

0       a
1       b
2       c
3    aaba
4    baca
5     NaN
6    caba
7     dog
8     cat
dtype: str

As well as simple manipulation, We can also match strings using [regular expressions](https://docs.nvidia.com/cudf/latest/cudf/api_docs/api/cudf.core.accessors.string.StringMethods.match/).

In [27]:
s.str.match("^[aAc].+")

0    False
1    False
2    False
3     True
4    False
5    False
6    False
7    False
8     True
dtype: bool

## Concat

Concatenating `Series` and `DataFrames` row-wise.

In [28]:
s = cudf.Series([1, 2, 3, None, 5])
cudf.concat([s, s])

0       1
1       2
2       3
3    <NA>
4       5
0       1
1       2
2       3
3    <NA>
4       5
dtype: int64

## Join

Performing SQL style merges. Note that the dataframe order is **not maintained**, but may be restored post-merge by sorting by the index.

In [29]:
df_a = cudf.DataFrame()
df_a["key"] = ["a", "b", "c", "d", "e"]
df_a["vals_a"] = [float(i + 10) for i in range(5)]

df_b = cudf.DataFrame()
df_b["key"] = ["a", "c", "e"]
df_b["vals_b"] = [float(i + 100) for i in range(3)]

merged = df_a.merge(df_b, on=["key"], how="left")
merged

,key,vals_a,vals_b
0,a,10.0,100.0
1,b,11.0,NaN
2,c,12.0,101.0
3,d,13.0,NaN
4,e,14.0,102.0


In [30]:
df["agg_col1"] = [1 if x % 2 == 0 else 0 for x in range(len(df))]
df["agg_col2"] = [1 if x % 3 == 0 else 0 for x in range(len(df))]

## Grouping

Grouping and then applying the `sum` function to the grouped data.

In [31]:
df.groupby("agg_col1").sum()

,a,b,c,agg_col2
agg_col1,,,,
1,90,100,90,4
0,100,90,100,3


Grouping hierarchically then applying the `sum` function to grouped data.

In [32]:
df.groupby(["agg_col1", "agg_col2"]).sum()

,,a,b,c
agg_col1,agg_col2,,,
1,1,36,40,36
0,0,73,60,73
1,0,54,60,54
0,1,27,30,27


Grouping and applying statistical functions to specific columns, using `agg`.

In [33]:
df.groupby("agg_col1").agg({"a": "max", "b": "mean", "c": "sum"})

,a,b,c
agg_col1,,,
1,18,10.0,90
0,19,9.0,100


## Transpose

In [34]:
sample = cudf.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})
sample

,a,b
0,1,4
1,2,5
2,3,6


In [35]:
sample.transpose()

,0,1,2
a,1,2,3
b,4,5,6


## Time Series

`DataFrames` supports `datetime` typed columns, which allow users to interact with and filter data based on specific timestamps.

In [36]:
import datetime as dt

date_df = cudf.DataFrame()
date_df["date"] = pd.date_range("11/20/2018", periods=72, freq="D")
date_df["value"] = cp.random.sample(len(date_df))

search_date = dt.datetime.strptime("2018-11-23", "%Y-%m-%d")
date_df.query("date <= @search_date")

,date,value
0,2018-11-20,0.986051
1,2018-11-21,0.232034
2,2018-11-22,0.397617
3,2018-11-23,0.103839


## Categoricals

`DataFrames` support categorical columns.

In [37]:
gdf = cudf.DataFrame(
    {"id": [1, 2, 3, 4, 5, 6], "grade": ["a", "b", "b", "a", "a", "e"]}
)
gdf["grade"] = gdf["grade"].astype("category")
gdf

,id,grade
0,1,a
1,2,b
2,3,b
3,4,a
4,5,a
5,6,e


In [38]:
gdf.grade.cat.categories

Index(['a', 'b', 'e'], dtype='str')

Accessing the underlying code values of each categorical observation.

In [39]:
gdf.grade.cat.codes

0    0
1    1
2    1
3    0
4    0
5    2
dtype: int8

## Converting to Pandas

In [40]:
df.head().to_pandas()

,a,b,c,agg_col1,agg_col2
0,0,19,0,1,1
1,1,18,1,0,0
2,2,17,2,1,0
3,3,16,3,0,1
4,4,15,4,1,0


## Converting to Numpy

In [41]:
df.to_numpy()

array([[ 0, 19,  0,  1,  1],
       [ 1, 18,  1,  0,  0],
       [ 2, 17,  2,  1,  0],
       [ 3, 16,  3,  0,  1],
       [ 4, 15,  4,  1,  0],
       [ 5, 14,  5,  0,  0],
       [ 6, 13,  6,  1,  1],
       [ 7, 12,  7,  0,  0],
       [ 8, 11,  8,  1,  0],
       [ 9, 10,  9,  0,  1],
       [10,  9, 10,  1,  0],
       [11,  8, 11,  0,  0],
       [12,  7, 12,  1,  1],
       [13,  6, 13,  0,  0],
       [14,  5, 14,  1,  0],
       [15,  4, 15,  0,  1],
       [16,  3, 16,  1,  0],
       [17,  2, 17,  0,  0],
       [18,  1, 18,  1,  1],
       [19,  0, 19,  0,  0]])

In [42]:
df["a"].to_numpy()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19])

## Converting to Arrow

In [43]:
df.to_arrow()

pyarrow.Table
a: int64
b: int64
c: int64
agg_col1: int64
agg_col2: int64
----
a: [[0,1,2,3,4,...,15,16,17,18,19]]
b: [[19,18,17,16,15,...,4,3,2,1,0]]
c: [[0,1,2,3,4,...,15,16,17,18,19]]
agg_col1: [[1,0,1,0,1,...,0,1,0,1,0]]
agg_col2: [[1,0,0,1,0,...,1,0,0,1,0]]

## Reading/Writing CSV Files

Writing to a CSV file.

In [44]:
if not os.path.exists("example_output"):
    os.mkdir("example_output")

df.to_csv("example_output/foo.csv", index=False)

Reading from a csv file.

In [45]:
df = cudf.read_csv("example_output/foo.csv")
df

,a,b,c,agg_col1,agg_col2
0,0,19,0,1,1
1,1,18,1,0,0
2,2,17,2,1,0
3,3,16,3,0,1
4,4,15,4,1,0
5,5,14,5,0,0
6,6,13,6,1,1
7,7,12,7,0,0
8,8,11,8,1,0
9,9,10,9,0,1


## Reading/Writing Parquet Files

Writing to parquet files with cuDF's GPU-accelerated parquet writer

In [46]:
df.to_parquet("example_output/temp_parquet")

Reading parquet files with cuDF's GPU-accelerated parquet reader.

In [47]:
df = cudf.read_parquet("example_output/temp_parquet")
df

,a,b,c,agg_col1,agg_col2
0,0,19,0,1,1
1,1,18,1,0,0
2,2,17,2,1,0
3,3,16,3,0,1
4,4,15,4,1,0
5,5,14,5,0,0
6,6,13,6,1,1
7,7,12,7,0,0
8,8,11,8,1,0
9,9,10,9,0,1


## Reading/Writing ORC Files

Writing ORC files.

In [48]:
df.to_orc("example_output/temp_orc")

And reading

In [49]:
df2 = cudf.read_orc("example_output/temp_orc")
df2

,a,b,c,agg_col1,agg_col2
0,0,19,0,1,1
1,1,18,1,0,0
2,2,17,2,1,0
3,3,16,3,0,1
4,4,15,4,1,0
5,5,14,5,0,0
6,6,13,6,1,1
7,7,12,7,0,0
8,8,11,8,1,0
9,9,10,9,0,1
